In [ ]:
# ============================================================
# CELL 9: Stage 3 — Load preference dataset
# ============================================================
# Format: {"prompt": "...", "chosen": "...", "rejected": "..."}
# ============================================================

def load_preference_dataset(jsonl_path: str) -> Dataset:
    """
    Loads the preference pairs for Stage 3 DPO alignment.

    DPO (Direct Preference Optimisation) teaches the model TASTE —
    which style of answer is preferred over which failure mode.

    Each example has 3 parts:
    - prompt:   the user question
    - chosen:   the GOOD answer (specific, safe, accurate)
    - rejected: the BAD answer (wrong, generic, incomplete, or unsafe)

    Your rejected answers have 5 different failure modes:
    1. Factually wrong (wrong company round counts)
    2. Too generic (no company/topic specificity)
    3. Incomplete (cuts off mid-advice)
    4. Unsafe (encourages resume dishonesty)
    5. Off-domain (answers something outside placement prep)

    Varied failure modes = model learns real preference, not just
    'avoid being rude' which is what happens with uniform rejection types.
    """
    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(
            f"File not found: {jsonl_path}\n"
            "Upload preference_dataset.jsonl to Colab."
        )

    raw = load_dataset("json", data_files=jsonl_path, split="train")

    required = {"prompt", "chosen", "rejected"}
    missing  = required - set(raw.column_names)
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {raw.column_names}")

    def clean_record(example):
        return {
            "prompt"  : example["prompt"].strip(),
            "chosen"  : example["chosen"].strip(),
            "rejected": example["rejected"].strip(),
        }

    dataset = raw.map(clean_record)

    print(f"✓ Preference dataset loaded: {len(dataset)} pairs")
    print(f"  Sample prompt  : {dataset[0]['prompt'][:80]}")
    print(f"  Sample chosen  : {dataset[0]['chosen'][:100]}")
    print(f"  Sample rejected: {dataset[0]['rejected'][:100]}")

    return dataset


stage3_dataset = load_preference_dataset(PREFERENCE_PATH)
print(f"\n✓ Stage 3 dataset ready: {len(stage3_dataset)} preference pairs")

In [ ]:
# ============================================================
# CELL 10: Stage 3 — DPO Preference Alignment
# Expected time: 10-15 minutes on T4
# Expected final loss: ~0.4 to 0.8 (DPO loss scale differs from SFT)
# ============================================================
# After this stage the model should be:
# - More professional in tone
# - Refusing to give unsafe advice (fake resume experience, etc.)
# - Giving specific answers, not generic ones
# - Staying on-domain (not answering medical/legal questions)
# ============================================================

print("=" * 55)
print("STAGE 3: DPO PREFERENCE ALIGNMENT")
print("Goal: Teach the model what NOT to say")
print(f"Starting from: Stage 2 merged model (knows how to answer Q&A)")
print("=" * 55)

# Load Stage 2 MERGED model
# This model already knows placement domain + how to answer questions.
# Stage 3 DPO refines its judgment — making chosen-style answers
# more likely and rejected-style answers less likely.
stage3_model, tokenizer = load_model_with_lora(STAGE2_MERGED)
FastLanguageModel.for_training(stage3_model)

# DPO uses LEFT padding (unlike SFT which uses right padding).
# Reason: DPO batches have variable-length prompts. Left padding ensures
# the attention mechanism focuses on real tokens at the RIGHT side.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

stage3_config = DPOConfig(
    output_dir  = f"{OUTPUT_ROOT}/stage3_logs",

    num_train_epochs            = STAGE3_EPOCHS,
    # 1 epoch only — DPO is very sensitive to over-training.
    # More than 1 epoch almost always causes 'reward collapse'
    # where the model becomes overconfident and repetitive.

    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,

    learning_rate  = STAGE3_LR,
    # 5e-5 — much lower than SFT stages.
    # DPO loss is computed as a ratio (log probability ratio of chosen vs rejected).
    # High LR with ratio-based loss = unstable training = model degrades.

    warmup_steps   = WARMUP_STEPS,
    logging_steps  = LOGGING_STEPS,

    save_strategy  = "no",
    report_to      = "none",

    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    optim = "adamw_8bit",

    beta = DPO_BETA,
    # beta=0.1: controls how strongly to push away from rejected responses.
    # Low beta (0.05) = gentle preference shift, less risk of degradation.
    # High beta (0.5) = aggressive, may collapse model quality.
    # 0.1 is the well-tested default from the original DPO paper.

    max_length        = MAX_SEQ_LENGTH,
    max_prompt_length = MAX_SEQ_LENGTH // 2,  # 512 tokens max for the prompt portion

    seed                 = SEED,
    remove_unused_columns= False,
    # remove_unused_columns=False: DPO trainer needs 'chosen' and 'rejected'
    # columns preserved — setting True would silently drop them.
)

stage3_trainer = DPOTrainer(
    model            = stage3_model,
    ref_model        = None,
    # ref_model=None: uses implicit reference from the model itself.
    # Standard approach — avoids needing 2x VRAM for a separate reference model.
    processing_class = tokenizer,
    train_dataset    = stage3_dataset,
    args             = stage3_config,
)

stage3_result = train_and_measure(stage3_trainer, "STAGE 3")

# ── Final 3-way evaluation ──────────────────────────────────────────────────
# These answers go into reports/final_evaluation.md
# Compare them with your Stage 2 answers from CELL 8 — you should see:
# - More professional tone
# - More specific company/topic details
# - Refusal on unsafe questions
tokenizer.padding_side = "right"  # switch back to right for inference

print("\n" + "="*55)
print("FINAL MODEL (ORPO/DPO) — SHOWCASE ANSWERS")
print("Copy into reports/final_evaluation.md")
print("="*55)

SHOWCASE_QUESTIONS = [
    "How should I prepare for Amazon SDE-1 placement?",
    "What DSA topics are most important for product company placements?",
    "I have a 6.5 CGPA. Can I still get into good companies?",
    "What is the TCS NQT exam pattern?",
    "How do I negotiate my first offer letter?",
]

for q in SHOWCASE_QUESTIONS:
    print(f"\nQ: {q}")
    ans = generate_answer(stage3_model, tokenizer, q, max_new_tokens=200)
    print(f"A: {ans}")
    print("-" * 40)

# Final save + merge — this is the model your Gradio demo loads
save_and_merge(
    model      = stage3_model,
    tokenizer  = tokenizer,
    adapter_dir= STAGE3_ADAPTER,
    merged_dir = FINAL_MERGED,
    stage_name = "Stage 3 Final",
)

del stage3_trainer
del stage3_model
clear_gpu_memory()

print("\n" + "="*55)
print("✓ ALL 3 STAGES COMPLETE")
print("="*55)